# 00load: load and tidy the KORAIL ridership files

**What it does.** Reads the raw KORAIL files in `data/raw`, turns them into tidy long tables, checks them against each other and against KORAIL's published totals, and saves the clean tables to `data/clean`.

**This part (sections 0 to 6)** covers the station tables:

- KTX boardings plus alightings by station and month, April 2004 to December 2022 (data.go.kr).
- KTX boardings and alightings by station, line by line, 2004 to 2015 (KORAIL statistics portal), used only as a check.
- All intercity trains by station and direction, KORAIL and SR, 2016 to 2024 (KORAIL statistics portal).
- KTX and SRT by station and direction, 2016 to 2024 (KORAIL statistics portal), which extends the KTX station series to 2024.

**Inputs:** `data/raw/datagokr/`, `data/raw/railstat/`.
**Outputs:** `data/clean/ktx_station_month.csv`, `data/clean/station_year_all.csv`, `data/clean/station_year_ktx_srt.csv`.

**Data credit:** Korea Railroad Corporation (KORAIL), via the Public Data Portal (data.go.kr) and the Korean Railway Statistics portal (railstat.korail.com). Free to use, no restrictions on scope of use.

## 0. Settings

In [1]:
from pathlib import Path
import io
import pandas as pd

RAW = Path("data/raw")  # <- folder with the downloads
CLEAN = Path("data/clean")  # <- folder for the tidy tables

FILE_KTX_MONTH = RAW / "datagokr" / "한국철도공사_KTX 월별 역별 승하차_20230101.csv"  # <- A1
FILE_KTX_0415 = RAW / "railstat" / "357_DT_357001_LD009_2004_20261029131129.csv"  # <- B2
FILE_ALL_1624 = RAW / "railstat" / "357_DT_357001_LD008_20261029130815.csv"  # <- B1 (all trains)
FILE_KTX_SRT_1624 = RAW / "railstat" / "역별_여객_승하차_실적(KTX_SRT)_20261029155701.csv"  # <- B1b (KTX and SRT)

ENC_DATAGOKR = "cp949"  # <- data.go.kr CSVs are saved in the Korean Windows encoding

CHECK_YEARS = range(2004, 2015)  # <- years where A1 and B2 should agree (B2 2015 is incomplete)
CHECK_TOL_PCT = 1.0  # <- largest acceptable difference between sources, in percent
CHECK_TOL_K = 20  # <- and in thousands of riders (small stations are rounded to thousands in A1)

# Stations renamed over time: old name in the portal tables -> name used in A1 (today's name)
STATION_RENAME = {"신경주": "경주", "신포항": "포항", "인천국제공항": "인천공항T1", "인천공항": "인천공항T1"}  # <-
OVERLAP_YEARS = range(2016, 2023)  # <- years covered by both A1 and B1b

# KORAIL's published yearly intercity riders (info.korail.com, passenger business page)
PUBLISHED_KORAIL = {2023: 136_693_000, 2024: 144_805_000}  # <- check figures

CLEAN.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)
print("pandas", pd.__version__)

pandas 3.1.0


## 1. Helpers

The statistics portal's long-format CSV exports end without a closing quote mark, so `read_portal` adds it before parsing. Codes are kept as text so leading zeros survive.

In [2]:
def read_portal(path):
    text = path.read_text(encoding="utf-8-sig")
    if text.count('"') % 2 == 1:
        text = text + '"'
    df = pd.read_csv(io.StringIO(text), dtype=str)
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
    return df


def to_num(s):
    s = s.str.replace(",", "", regex=False)
    s = s.replace({"-": None, "": None})
    return pd.to_numeric(s, errors="coerce")


def year_of(s):
    return s.str.extract(r"(\d{4})", expand=False).astype(int)


# Not real stations in the 2004 to 2015 line tables: junctions, links, signals, subtotals
NOT_STATION = r"계|연결|선$|신호|삼각|조차장"

## 2. KTX by station and month, 2004 to 2022 (A1)

One row per month, one column per station. Values are boardings plus alightings, in thousands, both directions together. Stations under 500 a month are shown as 0 by KORAIL.

In [3]:
a1 = pd.read_csv(FILE_KTX_MONTH, encoding=ENC_DATAGOKR)
print(a1.shape)
print(a1.iloc[:3, :8])

(225, 51)
        년월  검암  경산  경주  계룡  곡성  공주   광명
0  2004-04   0   0   0   0   0   0  126
1  2004-05   0   0   0   0   0   0  154
2  2004-06   0   0   0   0   0   0  139


In [4]:
ktx = a1.melt(id_vars="년월", var_name="station", value_name="riders_k")
ktx["month"] = pd.to_datetime(ktx["년월"] + "-01")
ktx = ktx.drop(columns="년월")
ktx = ktx[["month", "station", "riders_k"]]
ktx = ktx.sort_values(["station", "month"]).reset_index(drop=True)

first = ktx[ktx["riders_k"] > 0].groupby("station")["month"].min()
print("months:", ktx["month"].min().date(), "to", ktx["month"].max().date(), "|", ktx["month"].nunique())
print("stations:", ktx["station"].nunique())
print("first month with riders, latest openings:")
print(first.sort_values().tail(12).dt.strftime("%Y-%m").to_string())

months: 2004-04-01 to 2022-12-01 | 225
stations: 50
first month with riders, latest openings:
station
남원        2011-10
여천        2011-10
구례구       2011-10
함안        2012-12
진주        2012-12
경산        2013-01
인천공항T1    2014-06
검암        2014-07
포항        2015-04
공주        2015-04
인천공항T2    2018-01
서대구       2022-03


## 3. Check A1 against the portal's KTX table, 2004 to 2014 (B2)

B2 lists each station once per line it sits on, so a station's yearly total is the sum over lines of boardings plus alightings (강차 = alightings). Junctions and subtotal rows are removed. Only stations that appear in both sources are compared, and station-years with a zero month in A1 are skipped (the under-500 rule, or a station not yet open).

In [5]:
b2 = read_portal(FILE_KTX_0415)
b2.columns = ["code", "name", "dir_code", "direction", "period", "total", "board", "alight", "pkm"]
for col in ["total", "board", "alight", "pkm"]:
    b2[col] = to_num(b2[col])
b2["year"] = year_of(b2["period"])

st = b2[b2["code"].str.len() == 6]
st = st[~st["name"].str.contains(NOT_STATION)]
b2_year = st.groupby(["year", "name"])[["board", "alight"]].sum()
b2_year = (b2_year["board"] + b2_year["alight"]) / 1000
b2_year = b2_year.rename("b2_k").reset_index().rename(columns={"name": "station"})

ktx_y = ktx.assign(year=ktx["month"].dt.year, zero=ktx["riders_k"] == 0)
a1_year = ktx_y.groupby(["year", "station"]).agg(riders_k=("riders_k", "sum"), zero_months=("zero", "sum")).reset_index()
both = a1_year.merge(b2_year, on=["year", "station"], how="inner")
both = both[both["year"].isin(CHECK_YEARS)]
both = both[both["zero_months"] == 0]  # skip station-years hit by the under-500 rule or not yet open

chk = both.groupby("year")[["riders_k", "b2_k"]].sum()
chk["diff_pct"] = (chk["riders_k"] / chk["b2_k"] - 1) * 100
print(chk.round(1))
print()
print("largest station differences (thousands):")
both["diff_k"] = both["riders_k"] - both["b2_k"]
print(both.reindex(both["diff_k"].abs().sort_values(ascending=False).index).head(8).drop(columns="zero_months").round(1).to_string(index=False))
ok = chk["diff_pct"].abs().max() <= CHECK_TOL_PCT
print()
print("A1 agrees with B2 within", CHECK_TOL_PCT, "% every year:", ok)

      riders_k      b2_k  diff_pct
year                              
2004     37105   37105.4      -0.0
2005     63555   63553.1       0.0
2006     71556   71552.5       0.0
2007     73724   73806.6      -0.1
2008     75060   75059.1       0.0
2009     73747   73856.6      -0.1
2010     79613   79748.7      -0.2
2011     97387   97387.7      -0.0
2012    101273  100372.0       0.9
2013     99558   99560.1      -0.0
2014    103195  103205.0      -0.0

largest station differences (thousands):
 year station  riders_k    b2_k  diff_k
 2012    천안아산      5574  5327.5   246.5
 2012      서울     26942 26736.4   205.6
 2012      대전      9983  9843.7   139.3
 2012      광명      6877  6808.2    68.8
 2009      용산      4195  4247.9   -52.9
 2012      오송      1336  1284.8    51.2
 2012     동대구     12696 12650.2    45.8
 2007      부산     11085 11122.7   -37.7

A1 agrees with B2 within 1.0 % every year: True


## 4. All intercity trains by station, 2016 to 2024 (B1)

KORAIL (code 001) and SR (code 002) separately, one row per station, direction and year. This table covers every intercity train type, not only KTX. Its operator totals should match KORAIL's published yearly riders.

In [6]:
b1 = read_portal(FILE_ALL_1624)
b1.columns = ["code", "name", "dir_code", "direction", "period", "total", "board", "alight", "pkm"]
for col in ["total", "board", "alight", "pkm"]:
    b1[col] = to_num(b1[col])
b1["year"] = year_of(b1["period"])
b1["operator"] = b1["code"].str[:3].map({"001": "KORAIL", "002": "SR"})
b1["direction"] = b1["direction"].map({"하행": "down", "상행": "up"})

ops = b1[b1["code"].str.len() == 3]
stations = b1[b1["code"].str.len() == 6].copy()
stations = stations.rename(columns={"name": "station", "code": "station_code"})
stations = stations[["operator", "station_code", "station", "direction", "year", "board", "alight", "pkm"]]
print("stations:", stations.groupby("operator")["station"].nunique().to_dict())
print("years:", sorted(stations["year"].unique().tolist()))

stations: {'KORAIL': 339, 'SR': 32}
years: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [7]:
op_year = ops.groupby(["year", "operator"])["board"].sum().unstack()
st_year = stations.groupby(["year", "operator"])["board"].sum().unstack()
summary = pd.DataFrame(index=op_year.index)
summary["KORAIL_m"] = op_year["KORAIL"] / 1e6
summary["KORAIL_station_sum_m"] = st_year["KORAIL"] / 1e6
summary["SR_m"] = op_year["SR"] / 1e6
summary["published_KORAIL_m"] = pd.Series(PUBLISHED_KORAIL) / 1e6
summary["diff_vs_published_pct"] = (summary["KORAIL_m"] / summary["published_KORAIL_m"] - 1) * 100
print(summary.round(3))

      KORAIL_m  KORAIL_station_sum_m    SR_m  published_KORAIL_m  diff_vs_published_pct
year                                                                                   
2016   135.523               135.523   1.009                 NaN                    NaN
2017   127.842               127.842  19.467                 NaN                    NaN
2018   134.785               134.785  21.961                 NaN                    NaN
2019   139.531               139.531  23.329                 NaN                    NaN
2020    85.226                85.226  17.152                 NaN                    NaN
2021    93.536                93.536  19.563                 NaN                    NaN
2022   121.406               121.406  24.102                 NaN                    NaN
2023   136.693               136.693  26.412             136.693                    0.0
2024   144.805               144.805  26.690             144.805                   -0.0


## 5. KTX and SRT by station, 2016 to 2024 (B1b)

This export has the years across the top in three header rows (year, direction, measure), so it is rebuilt into one row per station, direction and year. Old station names are mapped to today's names (Section 0).

In [8]:
head = pd.read_csv(FILE_KTX_SRT_1624, encoding="utf-8-sig", header=None, dtype=str, nrows=3)
body = pd.read_csv(FILE_KTX_SRT_1624, encoding="utf-8-sig", header=None, dtype=str, skiprows=3)
print(head.iloc[:, :6])
print(body.shape)

         0        1                      2                      3                      4                                   5
0  A 역별(1)  A 역별(2)             Y2016 2016             Y2016 2016             Y2016 2016                          Y2016 2016
1  A 역별(1)  A 역별(2)  001 하행 (14STD04547 명)  001 하행 (14STD04547 명)  001 하행 (14STD04547 명)               001 하행 (14STD04547 명)
2  A 역별(1)  A 역별(2)                     소계              001001 승차              001002 하차  001003 인거리 (14357.KORAIL0014 null)
(119, 74)


In [9]:
def measure_of(label):
    if "승차" in label:
        return "board"
    if "하차" in label:
        return "alight"
    if "인거리" in label:
        return "pkm"
    return "total"


col_year = head.iloc[0, 2:].str[-4:].astype(int).tolist()
col_dir = ["down" if "하행" in x else "up" for x in head.iloc[1, 2:]]
col_meas = [measure_of(x) for x in head.iloc[2, 2:]]

values = body.iloc[:, 2:].apply(lambda c: to_num(c.astype("str")))
values.columns = pd.MultiIndex.from_arrays([col_year, col_dir, col_meas], names=["year", "direction", "measure"])
values = values.drop(columns="total", level="measure")
row_op = body[0].str.split(" ").str[1]
row_code = body[1].str.split(" ").str[0]
row_station = body[1].str.split(" ", n=1).str[1].fillna("subtotal")
values.index = pd.MultiIndex.from_arrays([row_op, row_code, row_station], names=["operator", "station_code", "station"])

kts = values.stack(["year", "direction"]).reset_index()
kts.columns.name = None

subtotal = kts[kts["station"] == "subtotal"]
kts = kts[kts["station"] != "subtotal"].copy()
kts["station_raw"] = kts["station"]
kts["station"] = kts["station"].replace(STATION_RENAME)
kts = kts[["operator", "station_code", "station", "station_raw", "direction", "year", "board", "alight", "pkm"]]

print("stations:", kts.groupby("operator")["station"].nunique().to_dict())
print("renamed:", sorted(kts.loc[kts["station"] != kts["station_raw"], "station_raw"].unique().tolist()))

stations: {'KTX': 81, 'SRT': 32}
renamed: ['신경주', '신포항', '인천공항', '인천국제공항']


In [10]:
by_op = kts.groupby(["year", "operator"])["board"].sum().unstack() / 1e6
by_sub = subtotal.groupby(["year", "operator"])["board"].sum().unstack() / 1e6
by_sub.columns = by_sub.columns + "_subtotal"
check = by_op.join(by_sub).round(2)
print("boardings, millions (station sum and the table's own subtotal):")
print(check)

boardings, millions (station sum and the table's own subtotal):
operator    KTX    SRT  KTX_subtotal  SRT_subtotal
year                                              
2016      64.62   1.01         64.62          1.01
2017      59.83  19.47         59.83         19.47
2018      67.04  21.96         67.04         21.96
2019      70.91  23.33         70.91         23.33
2020      43.56  17.15         43.56         17.15
2021      50.52  19.56         50.52         19.56
2022      70.80  24.10         70.80         24.10
2023      84.01  26.41         84.01         26.41
2024      89.89  26.69         89.89         26.69


## 6. Check B1b against A1, 2016 to 2022

Both are converted to boardings plus alightings in thousands per station and year. As in Section 3, station-years with a zero month in A1 are skipped. A difference is listed when it is larger than both tolerances (percent and thousands). Seoul is expected to differ from 2018: the portal table includes Gangneung line KTX riders at Seoul, while A1 covers only 50 stations and has none of the Gangneung line.

In [11]:
b1b_year = kts[kts["operator"] == "KTX"].groupby(["year", "station"])[["board", "alight"]].sum()
b1b_year = ((b1b_year["board"] + b1b_year["alight"]) / 1000).rename("b1b_k").reset_index()
cmp = a1_year.merge(b1b_year, on=["year", "station"], how="inner")
cmp = cmp[cmp["year"].isin(OVERLAP_YEARS)]
cmp = cmp[cmp["zero_months"] == 0]  # same rule as Section 3
cmp["diff_k"] = cmp["b1b_k"] - cmp["riders_k"]
cmp["diff_pct"] = cmp["diff_k"] / cmp["riders_k"] * 100
off = cmp[(cmp["diff_pct"].abs() > CHECK_TOL_PCT) & (cmp["diff_k"].abs() > CHECK_TOL_K)]
off = off.drop(columns="zero_months")

print("station-years compared:", len(cmp))
print("outside tolerance:")
print(off.round(1).to_string(index=False))
print()
print("stations outside tolerance:", sorted(set(off["station"])))
print("A1 stations missing from B1b:", sorted(set(a1_year["station"]) - set(b1b_year["station"])))

station-years compared: 283
outside tolerance:
 year station  riders_k   b1b_k  diff_k  diff_pct
 2018      서울     26093 28002.8  1909.8       7.3
 2019      서울     28752 31033.1  2281.1       7.9
 2020      서울     16729 18367.9  1638.9       9.8
 2021      서울     18396 20186.5  1790.5       9.7
 2022      서울     26756 29111.6  2355.6       8.8
 2022      행신      1768  1792.7    24.7       1.4

stations outside tolerance: ['서울', '행신']
A1 stations missing from B1b: ['광주', '구미', '김천', '함안']


## 7. Save

In [12]:
ktx.to_csv(CLEAN / "ktx_station_month.csv", index=False, encoding="utf-8-sig")
stations.to_csv(CLEAN / "station_year_all.csv", index=False, encoding="utf-8-sig")
kts.to_csv(CLEAN / "station_year_ktx_srt.csv", index=False, encoding="utf-8-sig")
for f in sorted(CLEAN.glob("*.csv")):
    print(f.name, f.stat().st_size // 1024, "KB")

ktx_station_month.csv 254 KB
station_year_all.csv 263 KB
station_year_ktx_srt.csv 115 KB


___
**END**